# LAB 1 — Query Rewriting: Reduzindo o Vocabulary Gap
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Implementar e comparar 3 técnicas de query rewriting em queries jurídicas reais,
medindo o impacto no retrieval antes e depois da reformulação.

**Entregáveis:**
- Pipeline de query rewriting funcional com 3 técnicas
- Tabela comparativa: query original vs reformulada para 5 queries do baseline
- Análise: qual técnica gerou maior diversidade semântica?

**Tempo estimado:** 40 minutos  
**Pré-requisito:** Aula 2 concluída — OpenSearch com corpus indexado


## ⚙️ Passo 1 — Instalação e Imports

In [29]:
# Instalar dependências
%pip install -q langchain langchain-community httpx langchain-openai langchain-ollama langchain-groq groq openai python-dotenv sentence-transformers opensearch-py pandas numpy
print("✅ Instalação concluída")

Note: you may need to restart the kernel to use updated packages.
✅ Instalação concluída


In [1]:
import os
import json
import time
import httpx
import pandas as pd
import numpy as np
from typing import List, Dict, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Carrega ~/mba-rag/.env (ou .env local) se existir
load_dotenv()

# ── Configurações Groq (primário) ─────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# ── Configurações Ollama (fallback local) ─────────────────────────
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "lama-3.1-8b")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3:latest")

# ── Configurações OpenSearch ──────────────────────────────────────
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
INDEX_NAME      = os.getenv("INDEX_NAME", "corpus_juridico")

print("✅ Configurações carregadas:")
print(f"   LLM primário:   Groq → {GROQ_LLM_MODEL}  ({'API key OK' if GROQ_API_KEY else 'SEM key → cairá no Ollama'})")
print(f"   LLM fallback:   Ollama → {OLLAMA_LLM_MODEL}  ({OLLAMA_BASE_URL})")
print(f"   Embeddings:     Ollama → {OLLAMA_EMBED_MODEL} (fallback: HuggingFace BAAI/bge-m3)")
print(f"   OpenSearch:     {OPENSEARCH_HOST}:{OPENSEARCH_PORT} | índice: {INDEX_NAME}")

✅ Configurações carregadas:
   LLM primário:   Groq → llama-3.1-8b-instant  (API key OK)
   LLM fallback:   Ollama → lama-3.1-8b  (http://localhost:11434)
   Embeddings:     Ollama → bge-m3:latest (fallback: HuggingFace BAAI/bge-m3)
   OpenSearch:     localhost:9200 | índice: corpus_juridico


## 🔌 Passo 2 — Conectar ao LLM e OpenSearch

In [2]:
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────
# Estratégia: Groq primário (llama-3.1-8b-instant) + Ollama fallback
# Ambos expõem API OpenAI-compatible, então usamos um único cliente.
# ─────────────────────────────────────────────────────────────────

llm_client = None
MODEL_NAME = None
LLM_PROVIDER = None

# Tentativa 1 — Groq (nuvem, baixa latência)
if GROQ_API_KEY:
    try:
        candidate = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY,http_client=httpx.Client(verify=False))
        _ = candidate.chat.completions.create(
            model=GROQ_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,

        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM via Groq | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); caindo para Ollama local…")

# Tentativa 2 — Ollama (fallback local, API OpenAI-compatible em /v1)
if llm_client is None:
    try:
        candidate = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        _ = candidate.chat.completions.create(
            model=OLLAMA_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,
        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, OLLAMA_LLM_MODEL, "ollama"
        print(f"✅ LLM via Ollama (fallback) | {OLLAMA_BASE_URL} | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"❌ Nenhum provedor LLM disponível: {e}")
        print("   Configure GROQ_API_KEY no .env ou rode `ollama serve` localmente.")


def llm_call(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    """Chama o LLM ativo (Groq ou Ollama) e retorna o texto gerado."""
    if llm_client is None:
        return ""
    try:
        resp = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️  LLM ({LLM_PROVIDER}) falhou: {e}")
        return ""


if llm_client is not None:
    test = llm_call("Responda em uma palavra: qual é a capital do Brasil?")
    print(f"   Smoke test → '{test}' (esperado: Brasília)")

✅ LLM via Groq | modelo: llama-3.1-8b-instant
   Smoke test → 'Brasília.' (esperado: Brasília)


In [3]:
from opensearchpy import OpenSearch

# Conectar ao OpenSearch
USE_OPENSEARCH = True  # se False, todo o pipeline usa fallback FAISS local

os_client = None
try:
    os_client = OpenSearch(
        hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
        http_auth=("admin", os.getenv("OPENSEARCH_PASSWORD", "admin")),
        use_ssl=False,
        verify_certs=False,
        timeout=10,
    )
    info = os_client.info()
    print(f"✅ OpenSearch conectado! Versão: {info['version']['number']}")
except Exception as e:
    print(f"⚠️  OpenSearch indisponível: {e}")
    print("   O lab cairá automaticamente em fallback FAISS local.")
    os_client = None
    USE_OPENSEARCH = False

✅ OpenSearch conectado! Versão: 3.0.0


## 📥 Passo 3 — Indexar o Corpus Jurídico

Antes de executar as técnicas de query rewriting, precisamos do corpus indexado para medir o
impacto no retrieval (Passo 7). Esta seção:

1. **Carrega o corpus** `corpus_juridico_aula3.json` (13 documentos: acórdãos STF/STJ + legislação CPP/CF)
2. **Gera embeddings BGE-M3** via Ollama (1024 dims, multilíngue) — fallback automático para HuggingFace `BAAI/bge-m3`
3. **Cria o índice kNN no OpenSearch** com mapping híbrido: campo textual em português (BM25) + campo vetorial (HNSW/cosseno)
4. **Indexa em bulk** — se OpenSearch estiver indisponível, monta um **vector store FAISS local** automaticamente

> ✅ Após este passo, o índice `corpus_juridico` (ou o FAISS em memória) está pronto para a busca híbrida do Passo 7.

In [4]:
# ── Embeddings BGE-M3: Ollama (primário) + HuggingFace (fallback) ─

EMBED_DIM = 1024  # BGE-M3 dimension
embeddings = None
EMBED_BACKEND = None

# Tenta Ollama (consistente com servidor LLM local da Aula 1)
try:
    from langchain_ollama import OllamaEmbeddings
    cand = OllamaEmbeddings(
        model=OLLAMA_EMBED_MODEL,
        base_url=OLLAMA_BASE_URL,
    )
    _ = cand.embed_query("teste de conexão")
    embeddings = cand
    EMBED_BACKEND = f"Ollama ({OLLAMA_EMBED_MODEL})"
    print(f"✅ Embeddings via Ollama: {OLLAMA_EMBED_MODEL}")
except Exception as e:
    print(f"⚠️  Ollama embeddings indisponível ({e}); tentando HuggingFace…")

# Fallback HuggingFace (BAAI/bge-m3 — mesmo espaço vetorial)
if embeddings is None:
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    EMBED_BACKEND = "HuggingFace (BAAI/bge-m3)"
    print(f"✅ Embeddings via HuggingFace (fallback): BAAI/bge-m3")

print(f"   Backend ativo: {EMBED_BACKEND} | dim={EMBED_DIM}")

d:\IBMEC\MBA_RAG_CAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Embeddings via Ollama: bge-m3:latest
   Backend ativo: Ollama (bge-m3:latest) | dim=1024


In [5]:
# ── Carregar corpus jurídico (acórdãos + legislação) ─────────────

DATASET_PATH = "../datasets/corpus_juridico_aula3.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

documentos = dataset["documentos"]
print(f"✅ Corpus carregado: {len(documentos)} documentos")
print()
print("📋 Amostra:")
for d in documentos[:3]:
    fonte = d.get("numero", d["id"])
    ementa = d.get("ementa", d.get("texto", ""))[:80]
    print(f"   [{d['id']}] {fonte} — {ementa}…")

✅ Corpus carregado: 80 documentos

📋 Amostra:
   [doc_001] HC 127.186/SP — HABEAS CORPUS. PRISÃO EM FLAGRANTE. TRÁFICO DE DROGAS. FLAGRANTE PREPARADO. AUSÊ…
   [doc_002] RHC 99.735/MG — PROCESSO PENAL. INTERROGATÓRIO. DIREITO AO SILÊNCIO. NEMO TENETUR SE DETEGERE. A…
   [doc_003] Lei 12.654/2012 — Prevê a coleta de perfil genético como forma de identificação criminal.…


In [6]:
# ── Criar índice OpenSearch com mapping híbrido (texto PT-BR + kNN) ─

INDEX_MAPPING = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 100,
        }
    },
    "mappings": {
        "properties": {
            "texto":          {"type": "text", "analyzer": "portuguese"},
            "ementa":         {"type": "text", "analyzer": "portuguese"},
            "palavras_chave": {"type": "keyword"},
            "tipo":           {"type": "keyword"},
            "tribunal":       {"type": "keyword"},
            "numero":         {"type": "keyword"},
            "data":           {"type": "date", "ignore_malformed": True},
            "embedding": {
                "type": "knn_vector",
                "dimension": EMBED_DIM,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {"ef_construction": 128, "m": 16},
                },
            },
        }
    },
}

if USE_OPENSEARCH and os_client is not None:
    try:
        if os_client.indices.exists(index=INDEX_NAME):
            print(f"ℹ️  Índice '{INDEX_NAME}' já existe — recriando para garantir mapping atualizado")
            os_client.indices.delete(index=INDEX_NAME)
        os_client.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
        print(f"✅ Índice '{INDEX_NAME}' criado com mapping kNN (HNSW/cosseno, dim={EMBED_DIM})")
    except Exception as e:
        print(f"⚠️  Falha ao criar índice OpenSearch: {e}")
        print("   Caindo para fallback FAISS local…")
        USE_OPENSEARCH = False
else:
    print("ℹ️  OpenSearch desabilitado — usando fallback FAISS direto")

ℹ️  Índice 'corpus_juridico' já existe — recriando para garantir mapping atualizado
✅ Índice 'corpus_juridico' criado com mapping kNN (HNSW/cosseno, dim=1024)


In [36]:
# ── Indexar documentos: gera embeddings + faz bulk no OpenSearch ──
# ── Fallback automático: monta vector store FAISS local             ──

from opensearchpy.helpers import bulk

faiss_store = None  # será preenchido apenas se OpenSearch falhar

# Pré-computa todos os embeddings de uma vez (1 chamada/doc no Ollama)
print(f"⏳ Gerando embeddings para {len(documentos)} documentos com {EMBED_BACKEND}…")
t0 = time.time()
textos_para_embed = [
    f"{d.get('ementa', '')}\n\n{d.get('texto', '')}".strip()
    for d in documentos
]
vetores = embeddings.embed_documents(textos_para_embed)
print(f"✅ {len(vetores)} embeddings gerados em {time.time()-t0:.1f}s")

if USE_OPENSEARCH and os_client is not None:
    actions = []
    for doc, vec in zip(documentos, vetores):
        actions.append({
            "_index": INDEX_NAME,
            "_id":    doc["id"],
            "_source": {
                "texto":          doc.get("texto", ""),
                "ementa":         doc.get("ementa", ""),
                "palavras_chave": doc.get("palavras_chave", []),
                "tipo":           doc.get("tipo", ""),
                "tribunal":       doc.get("tribunal", ""),
                "numero":         doc.get("numero", doc["id"]),
                "data":           doc.get("data", ""),
                "embedding":      vec,
            },
        })
    try:
        success, errors = bulk(os_client, actions, refresh="wait_for")
        print(f"✅ Bulk OpenSearch: {success} docs indexados; erros={len(errors) if isinstance(errors, list) else errors}")
        count = os_client.count(index=INDEX_NAME)["count"]
        print(f"   Total no índice '{INDEX_NAME}': {count}")
    except Exception as e:
        print(f"⚠️  Bulk OpenSearch falhou: {e}")
        print("   Caindo para FAISS local…")
        USE_OPENSEARCH = False

if not USE_OPENSEARCH:
    # Fallback FAISS local — mesmo espaço vetorial, sem servidor
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document as LCDocument

    lc_docs = [
        LCDocument(
            page_content=textos_para_embed[i],
            metadata={
                "id":     d["id"],
                "fonte":  d.get("numero", d["id"]),
                "tipo":   d.get("tipo", ""),
                "ementa": d.get("ementa", ""),
            },
        )
        for i, d in enumerate(documentos)
    ]
    faiss_store = FAISS.from_documents(lc_docs, embeddings)
    print(f"✅ FAISS local construído: {len(lc_docs)} documentos | dim={EMBED_DIM}")

print()
print(f"📊 Backend ativo: {'OpenSearch (' + INDEX_NAME + ')' if USE_OPENSEARCH else 'FAISS local (in-memory)'}")

⏳ Gerando embeddings para 13 documentos com Ollama (bge-m3:latest)…
✅ 13 embeddings gerados em 9.3s
✅ Bulk OpenSearch: 13 docs indexados; erros=0
   Total no índice 'corpus_juridico': 13

📊 Backend ativo: OpenSearch (corpus_juridico)


## 📋 Passo 4 — Carregar Queries do Baseline

In [37]:
# Carregar dataset com queries
DATASET_PATH = "../datasets/corpus_juridico_aula3.json"

try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    queries_baseline = dataset["queries_baseline"]
    print(f"✅ Dataset carregado: {len(queries_baseline)} queries")
except Exception:
    # Fallback: queries hardcoded para demonstração
    queries_baseline = [
        {"id": "q_001", "query_original": "Podem prender alguém sem mandado?"},
        {"id": "q_002", "query_original": "Policial pode ler mensagens do celular do suspeito?"},
        {"id": "q_003", "query_original": "O suspeito é obrigado a falar na delegacia?"},
        {"id": "q_004", "query_original": "O advogado pode ver o inquérito policial?"},
        {"id": "q_005", "query_original": "Como provar que o réu tinha intenção de matar?"},
    ]
    print("⚠️  Usando queries hardcoded (dataset não encontrado)")

print()
print("📋 Queries do baseline:")
for q in queries_baseline[:5]:
    print(f"  [{q['id']}] {q['query_original']}")

✅ Dataset carregado: 10 queries

📋 Queries do baseline:
  [q_001] Podem prender alguém sem mandado?
  [q_002] Policial pode ler mensagens do celular do suspeito?
  [q_003] O suspeito é obrigado a falar na delegacia?
  [q_004] O advogado pode ver o inquérito policial?
  [q_005] Como provar que o réu tinha intenção de matar?


## 🔄 Passo 5 — Implementar as 3 Técnicas de Rewriting

In [38]:
# ── TÉCNICA 1: Paraphrase Rewriting ──────────────────────────────

def paraphrase_rewriting(query: str, n: int = 3) -> List[str]:
    """Gera N reformulações técnicas da query jurídica."""
    prompt = f"""Você é especialista em direito processual penal brasileiro.
Reformule a query abaixo em {n} variações usando terminologia técnica jurídica.
Mantenha o mesmo significado, mas use linguagem técnica de documentos jurídicos.

Query original: {query}

Retorne APENAS as {n} variações, uma por linha, sem numeração ou introdução."""
    
    result = llm_call(prompt, temperature=0.5)
    if not result:
        return [query]  # fallback
    
    lines = [l.strip() for l in result.split('\n') if l.strip()]
    return lines[:n] if lines else [query]


# ── TÉCNICA 2: HyDE-Lite ─────────────────────────────────────────

def hyde_lite(query: str) -> str:
    """Gera parágrafo hipotético técnico para uso como query de busca."""
    prompt = f"""Você é redator jurídico especializado em processo penal brasileiro.
Escreva um parágrafo curto (4-6 linhas) que seria encontrado em acórdão ou código 
e que responderia diretamente à pergunta. Use linguagem técnica formal.

Pergunta: {query}

Escreva APENAS o parágrafo, sem título."""
    
    result = llm_call(prompt, temperature=0.2, max_tokens=300)
    return result if result else query


# ── TÉCNICA 3: Step-Back ─────────────────────────────────────────

def step_back(query: str) -> str:
    """Abstrai a query para nível conceitual mais geral."""
    prompt = f"""Você é especialista em direito processual penal.
Dada a pergunta específica abaixo, formule uma pergunta MAIS GERAL 
que abrange o conceito jurídico por trás dela.

Pergunta específica: {query}

Retorne APENAS a pergunta geral, sem explicação."""
    
    result = llm_call(prompt, temperature=0.1, max_tokens=100)
    return result if result else query

print("✅ Três técnicas de rewriting implementadas:")
print("   1. paraphrase_rewriting(query, n=3)")
print("   2. hyde_lite(query)")
print("   3. step_back(query)")

✅ Três técnicas de rewriting implementadas:
   1. paraphrase_rewriting(query, n=3)
   2. hyde_lite(query)
   3. step_back(query)


## 🧪 Passo 6 — Executar Rewriting nas 5 Queries

In [39]:
# Processar todas as queries do baseline
print("⏳ Processando queries com as 3 técnicas de rewriting...")
print("   (Isso pode levar 1-2 minutos dependendo do LLM)")
print()

resultados_rewriting = []

for q in queries_baseline[:5]:
    qid = q["id"]
    original = q["query_original"]
    print(f"Processando {qid}: '{original}'")
    
    t0 = time.time()
    
    # Aplicar as 3 técnicas
    paraphrases = paraphrase_rewriting(original, n=2)
    hyde_doc    = hyde_lite(original)
    stepback_q  = step_back(original)
    
    elapsed = time.time() - t0
    
    resultados_rewriting.append({
        "id": qid,
        "original": original,
        "paraphrase_1": paraphrases[0] if len(paraphrases) > 0 else original,
        "paraphrase_2": paraphrases[1] if len(paraphrases) > 1 else original,
        "hyde_lite": hyde_doc[:200] + "..." if len(hyde_doc) > 200 else hyde_doc,
        "step_back": stepback_q,
        "tempo_s": round(elapsed, 2)
    })
    
    print(f"  ✅ Concluído em {elapsed:.1f}s")

print()
print(f"✅ {len(resultados_rewriting)} queries processadas!")

⏳ Processando queries com as 3 técnicas de rewriting...
   (Isso pode levar 1-2 minutos dependendo do LLM)

Processando q_001: 'Podem prender alguém sem mandado?'
  ✅ Concluído em 1.4s
Processando q_002: 'Policial pode ler mensagens do celular do suspeito?'
  ✅ Concluído em 1.5s
Processando q_003: 'O suspeito é obrigado a falar na delegacia?'
  ✅ Concluído em 1.2s
Processando q_004: 'O advogado pode ver o inquérito policial?'
  ✅ Concluído em 1.3s
Processando q_005: 'Como provar que o réu tinha intenção de matar?'
  ✅ Concluído em 1.2s

✅ 5 queries processadas!


## 📊 Passo 7 — Tabela Comparativa

In [40]:
# Exibir tabela comparativa
print("📊 TABELA COMPARATIVA: Query Original vs Reformulações")
print("=" * 80)
print()

for r in resultados_rewriting:
    print(f"━━━ [{r['id']}] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"ORIGINAL:     {r['original']}")
    print(f"PARAPHRASE 1: {r['paraphrase_1']}")
    print(f"PARAPHRASE 2: {r['paraphrase_2']}")
    print(f"HYDE-LITE:    {r['hyde_lite'][:120]}...")
    print(f"STEP-BACK:    {r['step_back']}")
    print(f"Tempo:        {r['tempo_s']}s")
    print()

# Salvar como CSV para análise
df = pd.DataFrame(resultados_rewriting)
df.to_csv("/tmp/rewriting_comparativo.csv", index=False, encoding="utf-8")
print("✅ Resultados salvos em /tmp/rewriting_comparativo.csv")

📊 TABELA COMPARATIVA: Query Original vs Reformulações

━━━ [q_001] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ORIGINAL:     Podem prender alguém sem mandado?
PARAPHRASE 1: É possível a prisão preventiva sem o cumprimento do mandado de prisão preventiva?
PARAPHRASE 2: Pode ocorrer a detenção sem a prévia emissão de mandado de prisão?
HYDE-LITE:    A prisão em flagrante, prevista no artigo 302 do Código de Processo Penal, autoriza a detenção de pessoa que se encontra...
STEP-BACK:    Podem ser realizadas prisões preventivas sem mandado?
Tempo:        1.39s

━━━ [q_002] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ORIGINAL:     Policial pode ler mensagens do celular do suspeito?
PARAPHRASE 1: O agente policial pode realizar a busca e apreensão do aparelho móvel do investigado, o que inclui a leitura de suas mensagens, desde que com fundamento em ordem judicial ou autorização expressa do juiz competente.
PARAPHRASE 2: O agente policial pode proceder à busca e apreensão do aparelho móvel do investi

## 🔎 Passo 8 — Testar Impacto no Retrieval

In [41]:
def buscar_corpus(query: str, top_k: int = 5) -> List[Dict]:
    """Busca híbrida BM25 + kNN no corpus indexado.

    Estratégia OpenSearch:
        1. Gera o embedding da QUERY com o mesmo modelo (bge-m3) usado na indexação.
        2. `script_score` combina:
              - `multi_match` (BM25) em texto/ementa/palavras-chave (peso lexical 0.4)
              - `cosineSimilarity(embedding)` em campo `knn_vector` (peso semântico 0.6)
        3. Retorna os top_k documentos por score híbrido.

    Fallback FAISS: usa apenas similaridade vetorial (sem BM25, pois FAISS é puramente vetorial).
    """
    # ── OpenSearch: Hybrid Search BM25 + kNN ────────────────────────
    if USE_OPENSEARCH and os_client is not None:
        try:
            # Embedding da QUERY (mesmo modelo da indexação → mesmo espaço vetorial)
            q_vec = embeddings.embed_query(query)

            body = {
                "size": top_k,
                "query": {
                    "script_score": {
                        "query": {
                            "multi_match": {
                                "query":  query,
                                "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                                "type":   "best_fields",
                            }
                        },
                        "script": {
                            # Fusão linear: 0.4 BM25 normalizado + 0.6 cosseno (+1 para evitar negativos)
                            "source": "_score * 0.4 + cosineSimilarity(params.q, doc['embedding']) * 0.6 + 1.0",
                            "params": {"q": q_vec},
                        },
                    }
                },
            }
            resp = os_client.search(index=INDEX_NAME, body=body)
            return [
                {
                    "id":    h["_id"],
                    "score": round(h["_score"], 4),
                    "fonte": h["_source"].get("numero", h["_id"]),
                    "ementa": h["_source"].get("ementa", "")[:120],
                }
                for h in resp["hits"]["hits"]
            ]
        except Exception as e:
            print(f"⚠️  Hybrid search falhou ({e}); tentando FAISS…")

    # ── FAISS (fallback vetorial puro) ──────────────────────────────
    if faiss_store is not None:
        results = faiss_store.similarity_search_with_score(query, k=top_k)
        return [
            {
                "id":     d.metadata.get("id", ""),
                "score":  round(float(s), 4),
                "fonte":  d.metadata.get("fonte", d.metadata.get("id", "")),
                "ementa": d.metadata.get("ementa", "")[:120],
            }
            for d, s in results
        ]

    return []  # nenhum backend disponível

print("✅ Função buscar_corpus() pronta — hybrid BM25 + kNN no OpenSearch (fallback FAISS)")

✅ Função buscar_corpus() pronta — hybrid BM25 + kNN no OpenSearch (fallback FAISS)


### 🔬 Demo rápida — busca real para 1 query nas 4 variantes principais

In [42]:
# ── Demonstração rápida: busca para 1 query com as 4 variantes (original, hyde, stepback, paraphrase_1) ─

query_demo = resultados_rewriting[0]  # primeira query: q_001
qid = query_demo["id"]

print(f"🔬 Demo de busca para [{qid}]: \"{query_demo['original']}\"")
print("=" * 75)

variantes_demo = {
    "ORIGINAL":     query_demo["original"],
    "PARAPHRASE_1": query_demo["paraphrase_1"],
    "HYDE-LITE":    query_demo["hyde_lite"],
    "STEP-BACK":    query_demo["step_back"],
}

for nome, q in variantes_demo.items():
    print(f"\n▶ {nome}: {q[:80]}{'…' if len(q) > 80 else ''}")
    hits = buscar_corpus(q, top_k=5)
    for h in hits:
        print(f"    [{h['score']:.3f}] {h['fonte']:<30}  {h['ementa'][:60]}…")

print()
print(f"📊 Backend ativo: {'OpenSearch (hybrid BM25 + kNN)' if USE_OPENSEARCH else 'FAISS local (vetorial)'}")

🔬 Demo de busca para [q_001]: "Podem prender alguém sem mandado?"

▶ ORIGINAL: Podem prender alguém sem mandado?
    [2.086] RE 603.616/RO                   RECURSO EXTRAORDINÁRIO. INVIOLABILIDADE DO DOMICÍLIO. ENTRAD…

▶ PARAPHRASE_1: É possível a prisão preventiva sem o cumprimento do mandado de prisão preventiva…
    [5.382] Código de Processo Penal — Arts. 282-316  Medidas cautelares diversas da prisão e prisão preventiva.…
    [2.458] HC 127.186/SP                   HABEAS CORPUS. PRISÃO EM FLAGRANTE. TRÁFICO DE DROGAS. FLAGR…
    [2.253] RE 603.616/RO                   RECURSO EXTRAORDINÁRIO. INVIOLABILIDADE DO DOMICÍLIO. ENTRAD…
    [2.020] Resolução CNJ 213/2015          Dispõe sobre a apresentação de toda pessoa presa à autoridad…
    [1.856] Súmula Vinculante 14            É direito do defensor, no interesse do representado, ter ace…

▶ HYDE-LITE: A prisão em flagrante, prevista no artigo 302 do Código de Processo Penal, autor…
    [4.093] HC 127.186/SP                   HABE

---

## 📐 Passo 9 — Avaliação Quantitativa: Recall@5 e MRR por Técnica

Até aqui você implementou as 3 técnicas de rewriting e fez uma demo qualitativa.
Agora vamos **medir objetivamente** se cada técnica melhora o retrieval, comparando
contra o **ground truth** anotado manualmente no dataset.

### 🎯 O que vamos fazer

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│   PARA CADA UMA DAS 5 QUERIES DO BASELINE (q_001 a q_005):          │
│                                                                     │
│      ┌──────────────────────────────────────────────────────┐       │
│      │ Variantes geradas no Passo 6:                        │       │
│      │   1. original       (a pergunta do usuário, crua)    │       │
│      │   2. paraphrase_1   (reformulação técnica nº 1)      │       │
│      │   3. paraphrase_2   (reformulação técnica nº 2)      │       │
│      │   4. hyde_lite      (parágrafo hipotético jurídico)  │       │
│      │   5. step_back      (pergunta abstraída)             │       │
│      └──────────────────────────────────────────────────────┘       │
│                          │                                          │
│                          ▼                                          │
│      Cada variante é enviada ao buscar_corpus() definido            │
│      no Passo 8 → hybrid BM25 + kNN no OpenSearch                   │
│                          │                                          │
│                          ▼                                          │
│      Top-5 documentos retornados → comparamos com o                 │
│      ground truth (campo `documentos_relevantes` da query)          │
│                          │                                          │
│                          ▼                                          │
│      Calculamos 2 métricas:                                         │
│                                                                     │
│      • Recall@5 = (# relevantes nos top-5) / (# relevantes totais)  │
│                   ↑ "Quantos dos certos a busca trouxe?"            │
│                                                                     │
│      • MRR = 1/rank do PRIMEIRO doc relevante encontrado            │
│              ↑ "Quão perto do topo está o primeiro acerto?"         │
│                                                                     │
│   TOTAL: 5 queries × 5 variantes = 25 buscas no OpenSearch          │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 📚 Como interpretar as métricas

| Métrica | Faixa | Interpretação |
|---------|-------|---------------|
| **Recall@5** | 0.0 – 1.0 | `1.0` = trouxe TODOS os relevantes; `0.0` = não trouxe nenhum |
| **MRR**      | 0.0 – 1.0 | `1.0` = relevante já no 1º lugar; `0.5` = 2º lugar; `0.33` = 3º lugar; `0` = nenhum |

**Exemplo prático:** se a query tem 4 docs relevantes e a busca devolve 2 deles no top-5,
recall@5 = 2/4 = `0.50`. Se o primeiro relevante apareceu na posição 3, MRR = 1/3 ≈ `0.33`.

> 💡 **Recall** mede *quantidade* (cobertura); **MRR** mede *posição* (ranking).
> Um sistema pode ter alto recall mas baixo MRR (acerta mas embaixo da lista) — péssimo para o usuário.


### 🔹 Passo 9.1 — Carregar o Ground Truth do Corpus

O **ground truth** está no campo `documentos_relevantes` de cada query no arquivo
`corpus_juridico_aula3.json`. Cada query tem entre 7 e 11 documentos curados manualmente
como relevantes (acórdãos, leis, doutrina, laudos). Vamos transformar isso num
dicionário `{query_id: set(doc_ids_relevantes)}` para consulta rápida.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Carrega ground truth: dicionário {query_id → conjunto de doc_ids relevantes}
# ════════════════════════════════════════════════════════════════════
# O dataset traz em cada query o campo `documentos_relevantes` — uma lista
# de IDs (doc_001, doc_002, …) que sabemos serem respostas válidas para a query.
# Esses IDs foram curados manualmente por especialistas em direito penal.

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    _dataset_full = json.load(f)

ground_truth = {
    q["id"]: set(q.get("documentos_relevantes", []))
    for q in _dataset_full["queries_baseline"]
}

# Estatística do ground truth: quantos docs relevantes cada query tem?
print("📋 GROUND TRUTH POR QUERY")
print("=" * 70)
print(f"{'Query':<8} {'Nº rel.':<8}  Pergunta")
print("-" * 70)
for r in resultados_rewriting:
    rels = ground_truth.get(r["id"], set())
    print(f"  {r['id']:<6} {len(rels):>3}      {r['original'][:54]}")
print()
print(f"💡 Esses são os documentos que CONSIDERAMOS RELEVANTES para cada pergunta.")
print(f"   A busca acerta quando devolve um desses IDs nos top-5.")

### 🔹 Passo 9.2 — Executar a Matriz 5 Queries × 5 Variantes (25 buscas)

Para cada combinação `(query, variante)` vamos:

1. **Enviar a variante** ao `buscar_corpus()` (busca híbrida BM25 + kNN do Passo 8)
2. **Capturar os top-5 IDs** retornados
3. **Calcular Recall@5** = `|top5 ∩ ground_truth| / |ground_truth|`
4. **Calcular MRR** = `1 / posição_do_primeiro_relevante` (0 se nenhum acertou)
5. **Registrar a linha** num DataFrame para análise posterior

⏱️ **Tempo estimado:** 25 buscas × ~0.3s = ~8 segundos no OpenSearch
(ou ~15s se estiver usando o fallback FAISS).


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Constrói matriz completa: 5 queries × 5 variantes = 25 linhas
# ════════════════════════════════════════════════════════════════════

# Cada variante é UMA forma diferente de fazer a mesma pergunta:
variantes_nomes = ["original", "paraphrase_1", "paraphrase_2", "hyde_lite", "step_back"]
TOP_K = 5  # tamanho da lista de resultados (recall@K, MRR sobre os primeiros K)

linhas_matriz = []  # lista de dicionários — uma linha por (query × variante)

total_buscas = len(resultados_rewriting) * len(variantes_nomes)
print(f"⏳ Executando {len(resultados_rewriting)} queries × {len(variantes_nomes)} variantes = {total_buscas} buscas…")
print()

t0 = time.time()

for r in resultados_rewriting:                      # itera as 5 queries
    qid  = r["id"]                                  # ex: "q_001"
    rels = ground_truth.get(qid, set())             # ex: {"doc_001","doc_004",...}

    for v in variantes_nomes:                       # itera as 5 variantes
        texto_variante = r[v]                       # texto da reformulação

        # === A BUSCA SEMÂNTICA ACONTECE AQUI ===
        # buscar_corpus envia o texto ao OpenSearch:
        #   1) gera embedding bge-m3 do texto
        #   2) faz query híbrida: BM25 + cosineSimilarity(embedding) via script_score
        #   3) retorna top-5 docs ordenados por score híbrido
        hits = buscar_corpus(texto_variante, top_k=TOP_K)
        ids_recuperados = [h["id"] for h in hits]   # ex: ["doc_001","doc_017",...]

        # === MÉTRICA 1: RECALL@5 ===
        # Quantos dos relevantes (ground truth) apareceram nos top-5?
        true_positives = len(set(ids_recuperados) & rels)
        recall = true_positives / len(rels) if rels else 0.0

        # === MÉTRICA 2: MRR (Mean Reciprocal Rank) ===
        # Posição (1-indexada) do PRIMEIRO doc relevante na lista retornada
        # MRR = 1/rank. Se nenhum relevante apareceu → MRR = 0
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(ids_recuperados, start=1):
            if doc_id in rels:
                reciprocal_rank = 1.0 / rank
                break  # só o PRIMEIRO acerto importa para MRR

        linhas_matriz.append({
            "query_id":     qid,
            "variante":     v,
            "top5_ids":     ids_recuperados,
            "top5_fontes":  [h["fonte"] for h in hits],
            "ground_truth": sorted(rels),
            "true_pos":     true_positives,
            "recall@5":     round(recall, 3),
            "mrr":          round(reciprocal_rank, 3),
        })

elapsed = time.time() - t0
print(f"✅ Matriz construída em {elapsed:.1f}s ({len(linhas_matriz)} linhas, {elapsed/total_buscas*1000:.0f}ms/busca)")

# Constrói DataFrame para próximas análises
df_matriz = pd.DataFrame(linhas_matriz)
print()
print(f"📊 DataFrame `df_matriz` criado com shape {df_matriz.shape}")
print(f"   Colunas: {list(df_matriz.columns)}")

### 🔹 Passo 9.3 — Visão Geral: Tabela Colorida da Matriz Completa

A tabela abaixo mostra **todas as 25 buscas**. Cada linha é uma combinação
(query × variante). As colunas mostram:

- **`top5_fontes`**: as 5 fontes recuperadas (números de acórdão / lei)
- **`true_pos`**: quantos desses 5 estão no ground truth (verdadeiros positivos)
- **`recall@5`**: fração de relevantes que apareceram (`true_pos / |ground_truth|`)
- **`mrr`**: 1/(posição do primeiro relevante)

🎨 **Código de cores:**
- 🟢 **Verde claro** = `true_pos ≥ 3` (excelente — pelo menos 3 acertos)
- 🟡 **Amarelo**     = `true_pos == 1 ou 2` (razoável)
- 🔴 **Vermelho**    = `true_pos == 0` (nenhum acerto, a variante errou todos)


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Renderização colorida da matriz com pandas Styler
# ════════════════════════════════════════════════════════════════════

def colorir_acertos(row):
    """Pinta a linha conforme o nº de acertos (true_pos)."""
    n = row["true_pos"]
    if n >= 3:
        cor = "background-color: #c8e6c9"        # verde claro
    elif n >= 1:
        cor = "background-color: #fff9c4"        # amarelo
    else:
        cor = "background-color: #ffcdd2"        # vermelho claro
    return [cor] * len(row)

# Versão compacta da tabela para visualização (oculta listas longas)
df_visu = df_matriz[["query_id", "variante", "true_pos", "recall@5", "mrr", "top5_fontes"]].copy()
df_visu["top5_fontes"] = df_visu["top5_fontes"].apply(lambda x: " | ".join(x))

styled = (
    df_visu
    .style
    .apply(colorir_acertos, axis=1)
    .format({"recall@5": "{:.2f}", "mrr": "{:.2f}"})
    .set_properties(**{"font-size": "11px", "text-align": "left"})
    .set_table_styles([{"selector": "th",
                        "props": [("background-color", "#37474f"),
                                  ("color", "white"),
                                  ("font-weight", "bold"),
                                  ("text-align", "left")]}])
)

print("🎨 MATRIZ COLORIDA — 5 queries × 5 variantes")
print("   🟢 verde = ≥3 acertos | 🟡 amarelo = 1-2 acertos | 🔴 vermelho = 0 acertos")
print()
styled  # exibe diretamente no Colab/Jupyter

### 🔹 Passo 9.4 — Agregação: Métrica Média por Técnica

Agora consolidamos as 25 linhas da matriz em **5 linhas — uma por técnica de rewriting**.
Para cada técnica calculamos a **média das 5 queries** em Recall@5 e MRR.

> 💡 Esta é a tabela que responde à pergunta principal do lab:
> **"Qual técnica de query rewriting deu o melhor retrieval?"**


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Agregação: média por técnica sobre as 5 queries
# ════════════════════════════════════════════════════════════════════
# groupby("variante") agrupa as 25 linhas em 5 grupos (um por técnica),
# depois calculamos a média de cada métrica dentro de cada grupo.

agregado = (
    df_matriz
    .groupby("variante", as_index=False)
    .agg(recall_5_medio=("recall@5", "mean"),
         mrr_medio=("mrr", "mean"),
         tp_total=("true_pos", "sum"))
    .sort_values("recall_5_medio", ascending=False)
    .reset_index(drop=True)
)
agregado["recall_5_medio"] = agregado["recall_5_medio"].round(3)
agregado["mrr_medio"]      = agregado["mrr_medio"].round(3)

# Adiciona rank visual (1º, 2º, 3º, 4º, 5º)
medalhas = ["🥇", "🥈", "🥉", "  4º", "  5º"]
agregado.insert(0, "rank", medalhas[:len(agregado)])

print("📊 MÉTRICAS AGREGADAS POR TÉCNICA")
print("   (média sobre as 5 queries do baseline)")
print("=" * 65)
print(agregado.to_string(index=False))
print()

# Captura os baselines para comparação relativa
recall_original = agregado.loc[agregado["variante"] == "original", "recall_5_medio"].values[0]
mrr_original    = agregado.loc[agregado["variante"] == "original", "mrr_medio"].values[0]

print(f"📍 BASELINE (query original sem rewriting):")
print(f"   recall@5 = {recall_original:.3f}")
print(f"   MRR      = {mrr_original:.3f}")
print()
print("📈 GANHO/PERDA RELATIVO À QUERY ORIGINAL:")
for _, row in agregado.iterrows():
    if row["variante"] == "original":
        continue
    ganho_r = (row["recall_5_medio"] - recall_original) * 100  # em pontos percentuais
    ganho_m =  row["mrr_medio"]      - mrr_original

    seta_r = "↑" if ganho_r > 0 else ("↓" if ganho_r < 0 else "=")
    seta_m = "↑" if ganho_m > 0 else ("↓" if ganho_m < 0 else "=")

    print(f"   {row['variante']:<14} recall {seta_r} {ganho_r:+5.1f} pp   |   MRR {seta_m} {ganho_m:+.3f}")
print()
print("ℹ️  'pp' = pontos percentuais (ex: +10 pp = subiu 10% no recall absoluto)")

### 🔹 Passo 9.5 — Visualização Gráfica: Barplot Recall@5 e MRR

O gráfico abaixo mostra lado a lado:

- **Esquerda:** Recall@5 médio por técnica (com linha do baseline `original` tracejada)
- **Direita:** MRR médio por técnica

🟢 Barras **verdes** = variantes de rewriting | 🔵 Barra **azul** = baseline `original`

Procure visualmente: **qual barra verde supera a linha azul tracejada?** Essa é a
técnica que melhorou de fato — para esta combinação de dataset e LLM.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Gráfico comparativo: 2 barplots lado a lado
# ════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x      = np.arange(len(agregado))
bar_w  = 0.6
# Cores: azul para baseline, verde para variantes que melhoraram
cores  = ["#1f77b4" if v == "original" else "#2ca02c" for v in agregado["variante"]]

# ── Subplot 1: Recall@5 ──────────────────────────────────────────────
axes[0].bar(x, agregado["recall_5_medio"], width=bar_w, color=cores, edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels(agregado["variante"], rotation=20, ha="right")
axes[0].set_ylabel("Recall@5 médio")
axes[0].set_title("Recall@5 por Técnica de Rewriting", fontweight="bold")
axes[0].set_ylim(0, max(agregado["recall_5_medio"].max() * 1.25, 0.05))
axes[0].axhline(y=recall_original, color="#1f77b4", linestyle="--", linewidth=1, alpha=0.6,
                label=f"baseline original ({recall_original:.2f})")
for i, v in enumerate(agregado["recall_5_medio"]):
    axes[0].text(i, v + 0.005, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

# ── Subplot 2: MRR ───────────────────────────────────────────────────
axes[1].bar(x, agregado["mrr_medio"], width=bar_w, color=cores, edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(agregado["variante"], rotation=20, ha="right")
axes[1].set_ylabel("MRR médio")
axes[1].set_title("MRR por Técnica de Rewriting", fontweight="bold")
axes[1].set_ylim(0, max(agregado["mrr_medio"].max() * 1.25, 0.05))
axes[1].axhline(y=mrr_original, color="#1f77b4", linestyle="--", linewidth=1, alpha=0.6,
                label=f"baseline original ({mrr_original:.2f})")
for i, v in enumerate(agregado["mrr_medio"]):
    axes[1].text(i, v + 0.005, f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

# ── Título global ────────────────────────────────────────────────────
backend_label = "OpenSearch (hybrid BM25 + kNN)" if USE_OPENSEARCH else "FAISS local (vetorial)"
plt.suptitle(
    f"Impacto do Query Rewriting no Retrieval — Backend: {backend_label}",
    fontsize=11, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("/tmp/lab1_metricas_rewriting.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"✅ Gráfico salvo em /tmp/lab1_metricas_rewriting.png")

---

## 🔬 Passo 10 — Análise Qualitativa: Drill-down por Query

A média do Passo 9 esconde detalhes. Vamos olhar **query-por-query**: em quais
casos cada técnica brilhou? Em quais ela atrapalhou?

> Esta análise é o que você levaria para uma reunião com o cliente jurídico:
> "no caso do flagrante, paraphrase funcionou; no caso do silêncio, HyDE atrapalhou".


### 🔹 Passo 10.1 — Tabela Cruzada: Recall por (Query × Técnica)

Pivota a matriz: cada linha é uma query, cada coluna é uma técnica.
A célula mostra o recall@5 daquela combinação.

Olhe linha por linha: **qual técnica venceu naquela query específica?**


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Pivot: query × variante → matriz de recall
# ════════════════════════════════════════════════════════════════════
pivot_recall = df_matriz.pivot(index="query_id", columns="variante", values="recall@5")

# Reordena colunas para padrão original primeiro
ordem_cols = ["original", "paraphrase_1", "paraphrase_2", "hyde_lite", "step_back"]
pivot_recall = pivot_recall[ordem_cols]

# Adiciona coluna "vencedor" (qual técnica teve maior recall para cada query)
pivot_recall["🏆 vencedor"] = pivot_recall[ordem_cols].idxmax(axis=1)
pivot_recall["Δ vs orig"]   = (pivot_recall[ordem_cols].max(axis=1) - pivot_recall["original"]).round(3)

# Styler: destaca em verde o MAIOR valor de cada linha
def highlight_max_row(row):
    is_max = [False] * len(row)
    # só compara as 5 colunas numéricas (ignora 🏆 vencedor e Δ)
    cols_num = [c for c in row.index if c in ordem_cols]
    valores  = [row[c] for c in cols_num]
    if valores:
        max_val = max(valores)
        is_max  = [(c in cols_num and row[c] == max_val) for c in row.index]
    return ["background-color: #c8e6c9; font-weight: bold" if b else "" for b in is_max]

styled_pivot = (
    pivot_recall
    .style
    .apply(highlight_max_row, axis=1)
    .format({c: "{:.2f}" for c in ordem_cols + ["Δ vs orig"]})
    .set_caption("Recall@5 por Query × Técnica (🟢 = melhor técnica daquela query)")
    .set_table_styles([{"selector": "th",
                        "props": [("background-color", "#37474f"),
                                  ("color", "white")]}])
)

print("📊 TABELA CRUZADA — Recall@5 por (Query × Técnica)")
print("   Cada linha = uma pergunta. Verde = técnica vencedora daquela linha.")
print()
styled_pivot

### 🔹 Passo 10.2 — Inspeção Manual: Por Que a Técnica Vencedora Ganhou?

Pegue a query que teve o **maior ganho relativo** ao usar rewriting e olhe
**exatamente quais documentos** cada variante trouxe. Isso revela se a
melhoria veio de mais cobertura semântica ou de melhor ranking.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Inspeção manual: query com maior ganho vs original
# ════════════════════════════════════════════════════════════════════
query_destaque = pivot_recall["Δ vs orig"].idxmax()
melhor_tec     = pivot_recall.loc[query_destaque, "🏆 vencedor"]
ganho          = pivot_recall.loc[query_destaque, "Δ vs orig"]

print(f"🔍 QUERY COM MAIOR GANHO: {query_destaque}")
print(f"   Técnica vencedora: '{melhor_tec}' (+{ganho:.2f} em recall@5)")
print("=" * 75)
print()

# Encontra a linha do resultados_rewriting correspondente
r_destaque = next(r for r in resultados_rewriting if r["id"] == query_destaque)
gt_destaque = ground_truth.get(query_destaque, set())

print(f"❓ Pergunta original:  {r_destaque['original']}")
print(f"📋 Ground truth ({len(gt_destaque)} docs): {sorted(gt_destaque)}")
print()

# Para cada variante, mostra: texto + top-5 com ✓/✗ ao lado
for v in ordem_cols:
    texto = r_destaque[v]
    print("─" * 75)
    print(f"▶ VARIANTE: {v.upper()}")
    print(f"  texto: \"{texto[:100]}{'…' if len(texto) > 100 else ''}\"")

    # Recupera essa linha específica da matriz
    linha = df_matriz[(df_matriz["query_id"] == query_destaque) &
                       (df_matriz["variante"] == v)].iloc[0]
    print(f"  recall@5 = {linha['recall@5']:.2f}  |  MRR = {linha['mrr']:.2f}  |  acertos = {linha['true_pos']}")
    print("  TOP-5 retornados:")
    for rank, (doc_id, fonte) in enumerate(zip(linha["top5_ids"], linha["top5_fontes"]), start=1):
        marker = "✅" if doc_id in gt_destaque else "❌"
        print(f"    {rank}. {marker} [{doc_id}]  {fonte}")
    print()

---

## ✅ Passo 11 — Síntese Final e Conclusões

Vamos consolidar tudo num resumo executivo: o que aprendemos sobre query rewriting
neste corpus jurídico de 80 documentos.


In [ ]:
# ════════════════════════════════════════════════════════════════════
# Síntese automática baseada nas métricas calculadas
# ════════════════════════════════════════════════════════════════════

melhor_tec_global = agregado.iloc[0]["variante"]
pior_tec_global   = agregado.iloc[-1]["variante"]
delta_recall      = (agregado.iloc[0]["recall_5_medio"] - recall_original) * 100
delta_mrr         =  agregado.iloc[0]["mrr_medio"]      - mrr_original

print("📋 RESUMO EXECUTIVO — LAB 1: Query Rewriting")
print("=" * 70)
print()
print("📊 CONFIGURAÇÃO DO EXPERIMENTO:")
print(f"   • Corpus jurídico:    80 documentos (52 relevantes + 28 distractors)")
print(f"   • Queries testadas:   {len(resultados_rewriting)} (q_001 a q_005)")
print(f"   • Variantes por query: {len(variantes_nomes)} (original + 4 rewrites)")
print(f"   • Total de buscas:    {len(linhas_matriz)}")
print(f"   • Backend retrieval:  {'OpenSearch (hybrid BM25 + kNN)' if USE_OPENSEARCH else 'FAISS local (vetorial)'}")
print(f"   • LLM rewriter:       {LLM_PROVIDER} ({MODEL_NAME})")
print(f"   • Embeddings:         {EMBED_BACKEND}")
print()
print("🏆 RANKING POR RECALL@5 MÉDIO:")
for i, row in agregado.iterrows():
    flag = ["🥇","🥈","🥉","  ","  "][i] if i < 5 else "  "
    print(f"   {flag} {i+1}. {row['variante']:<14} recall={row['recall_5_medio']:.3f}  mrr={row['mrr_medio']:.3f}")
print()
print("📈 BASELINE vs MELHOR TÉCNICA:")
print(f"   Baseline (original):     recall={recall_original:.3f}  mrr={mrr_original:.3f}")
print(f"   Melhor ({melhor_tec_global}):       recall={agregado.iloc[0]['recall_5_medio']:.3f}  mrr={agregado.iloc[0]['mrr_medio']:.3f}")
print(f"   Ganho:                   recall {delta_recall:+.1f} pp  |  mrr {delta_mrr:+.3f}")
print()
print("─" * 70)
print("EXPORTAÇÃO DOS RESULTADOS:")
print("─" * 70)

# Exportar CSV com top5_ids/fontes serializados
df_export = df_matriz.copy()
df_export["top5_ids"]     = df_export["top5_ids"].apply(lambda x: ";".join(x))
df_export["top5_fontes"]  = df_export["top5_fontes"].apply(lambda x: ";".join(x))
df_export["ground_truth"] = df_export["ground_truth"].apply(lambda x: ";".join(x))

df_export.to_csv("/tmp/lab1_matriz_buscas.csv", index=False, encoding="utf-8")
agregado.to_csv("/tmp/lab1_metricas_agregadas.csv", index=False, encoding="utf-8")
pivot_recall.to_csv("/tmp/lab1_pivot_recall.csv", encoding="utf-8")

print(f"   /tmp/lab1_matriz_buscas.csv         ← 25 linhas (query × variante)")
print(f"   /tmp/lab1_metricas_agregadas.csv    ← 5 linhas (média por técnica)")
print(f"   /tmp/lab1_pivot_recall.csv          ← matriz cruzada query × técnica")
print(f"   /tmp/lab1_metricas_rewriting.png    ← gráfico de barras")
print()
print("✅ LAB 1 CONCLUÍDO!")

### 📋 Checklist de Entrega

- [x] **Embeddings BGE-M3** funcionando (Ollama ou HuggingFace fallback)
- [x] **Corpus indexado** no OpenSearch com mapping kNN (ou fallback FAISS)
- [x] **3 técnicas de rewriting** implementadas (paraphrase, HyDE-lite, step-back)
- [x] **Hybrid search BM25 + kNN** via `script_score` no OpenSearch
- [x] **Matriz 5×5** executada (25 buscas) com tracking de top-5 IDs
- [x] **Recall@5 e MRR** calculados contra ground truth
- [x] **Tabela colorida** mostrando acertos por (query × variante)
- [x] **Pivot table** com técnica vencedora por query
- [x] **Drill-down** mostrando ✅/❌ doc-a-doc na query de maior ganho
- [x] **Barplot** comparando técnicas
- [ ] **Reflexão escrita** pelo aluno (espaço abaixo)

---

## 📝 Espaço para Reflexão do Aluno

*Responda com base nos números que você acabou de gerar:*

**1. Qual técnica venceu no recall@5 médio? Por uma diferença significativa ou apertada?**
→ *[sua resposta]*

**2. Olhe a pivot table do Passo 10.1: alguma técnica teve desempenho consistente
em TODAS as 5 queries, ou houve variação grande?**
→ *[sua resposta]*

**3. No drill-down do Passo 10.2, identifique 1 documento que apareceu na variante
vencedora mas NÃO na original. Por que ele foi recuperado?**
→ *[sua resposta]*

**4. O HyDE-Lite teve algum caso onde o parágrafo hipotético "alucinou" um termo
que levou a docs irrelevantes? Como você mitigaria isso em produção jurídica?**
→ *[sua resposta]*

**5. Custo×benefício: cada query rewriting consome ~100-300 tokens do Groq
(`llama-3.1-8b-instant`). Para um sistema com 1.000 queries/dia, o ganho de recall
justifica esse custo? Em quais cenários você desligaria o rewriting?**
→ *[sua resposta]*

---

### 🚀 Próximo Passo (LAB 2)

Combine este pipeline com **BGE-Reranker** (cross-encoder) para reordenar os top-K
recuperados. Você verá o ganho *multiplicativo* de Advanced RAG completo:
rewriting (Passo 9 aqui) + retrieval híbrido (Passo 8) + reranking (LAB 2) =
Recall@5 e MRR ainda maiores.
